In [8]:
def num_pad(source, target):
        crxb_index = math.ceil(source / target)
        num_padding = crxb_index * target - source
        return crxb_index, num_padding

def num_pad_col(source, target, quantization_resolution):
    #crxb_col_fit = target/(quantization_resolution/2)
    crxb_index = math.ceil((source * (quantization_resolution/2)) / target)
    num_padding =int( crxb_index * target / (quantization_resolution/2) - source)
    return crxb_index, num_padding
def quantize_dac(input, delta_x):
    output = torch.round(input / delta_x)
    return output
quantize_weight = quantize_dac
def x_relu(input):
    return input.clamp(min=0)
       
def bitslicer(input_tensor):
    # Ensure the new tensors created are on the same device as the input tensor
    device = input_tensor.device
    
    size = input_tensor.size()
    tensor_int16 = input_tensor.to(torch.int16)

    # Create a list to hold the 2-bit segments
    segments = []
    for index in range(7, -1, -1):
        shifted_tensor = tensor_int16 >> (2 * index)
        segment = shifted_tensor & 0b11
        # Make sure the segment is created on the correct device
        segments.append(segment.to(device))

    # Stack the segments along a new dimension and ensure it's on the correct device
    segments_tensor = torch.stack(segments, dim=-1).to(device).float()

    # Reshape the segments tensor to the desired shape
    reshaped_tensor = segments_tensor.transpose(3, 4).reshape(size[0], size[1], -1, size[3])

    return reshaped_tensor


def create_masks_for_conv_layers(model, crxb_size=128, quantization_resolution=16):
    masks_list = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            weight = module.weight.data  # Extract the weights from the conv layer
            
            # Following the quantization and mask creation process
            n_lvl_w = 2 ** quantization_resolution
            h_lvl_w = (n_lvl_w - 2) / 2
            delta_w = weight.abs().max() / h_lvl_w
            
            # Quantize weight
            weight_quan = quantize_dac(weight, delta_w)
            
            # Flatten and pad weights
            weight_flatten = weight_quan.view(module.out_channels, -1)
            crxb_row, crxb_row_pads = num_pad(weight_flatten.shape[1], crxb_size)
            crxb_col, crxb_col_pads = num_pad_col(weight_flatten.shape[0], crxb_size, quantization_resolution)
            w_pad = (0, crxb_row_pads, 0, crxb_col_pads)
            weight_padded = F.pad(weight_flatten, w_pad, mode='constant', value=0)
            
            # View and transpose weights for crossbar
            col_size = int(crxb_size / (quantization_resolution / 2))
            weight_crxb = weight_padded.view(crxb_col, col_size, crxb_row, crxb_size).transpose(1, 2)
            
            # Generate masks
            positive_tensor = x_relu(weight_crxb)
            # negative_tensor = x_relu(-weight_crxb)
            input_scaled_pos = bitslicer(positive_tensor)
            # input_scaled_neg = bitslicer(negative_tensor)
            positive_mask = input_scaled_pos > 0
            # negative_mask = ~positive_mask
            
            # Append masks to the list
            masks_list.append(positive_mask)
        if isinstance(module, nn.Linear):
            weight = module.weight.data
            # Following the quantization and mask creation process
            n_lvl_w = 2 ** quantization_resolution
            h_lvl_w = (n_lvl_w - 2) / 2
            delta_w = weight.abs().max() / h_lvl_w
            
            # Quantize weight
            weight_quan = quantize_dac(weight, delta_w)
            # Flatten and pad weights
            # weight_flatten = weight_quan.view(module.out_channels, -1)
            crxb_row, crxb_row_pads = num_pad(weight_quan.shape[1], crxb_size)
            crxb_col, crxb_col_pads = num_pad_col(weight_quan.shape[0], crxb_size, quantization_resolution)
            w_pad = (0, crxb_row_pads, 0, crxb_col_pads)
            weight_padded = F.pad(weight_quan, w_pad, mode='constant', value=0)
            # View and transpose weights for crossbar
            col_size = int(crxb_size / (quantization_resolution / 2))
            weight_crxb = weight_padded.view(crxb_col, col_size, crxb_row, crxb_size).transpose(1, 2)
             # Generate masks
            positive_tensor = x_relu(weight_crxb)
            # negative_tensor = x_relu(-weight_crxb)
            input_scaled_pos = bitslicer(positive_tensor)
            # input_scaled_neg = bitslicer(negative_tensor)
            positive_mask = input_scaled_pos > 0
            # negative_mask = ~positive_mask
            
            # Append masks to the list
            masks_list.append(positive_mask)
    
    return masks_list


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import time
import os
import sys
import copy
import math
import matplotlib.pyplot as plt
from torx.module.layer import *
from benchmark import replace_layers
import argparse
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vgg16
# from torchsummary import summary
from resnet import *
from mobilenet import *
from VGG import *
from densenet import * 
import json

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# model = ResNet18(num_classes=10)
# model = VGG('VGG16',num_classes=10)
model = densenet121(num_classes=10)
model.load_state_dict(torch.load("trained_models/densenet121_cifar10.pth"))
model.to(device)

masks_list = create_masks_for_conv_layers(model, crxb_size=128, quantization_resolution=16)

In [13]:
for i in range(len(masks_list)):
    print(masks_list[i].shape)

torch.Size([4, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 2, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 3, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 3, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 3, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 3, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8, 4, 128, 128])
torch.Size([2, 9, 128, 128])
torch.Size([8,

running experiment with fault rate and model:  0.1 resnet18
Run 1/1

Test set: Average loss: 2.5574, Accuracy: 3431/10000 (34.3100%)
running experiment with fault rate and model:  0.5 resnet18
Run 1/1

Test set: Average loss: 4.0114, Accuracy: 1475/10000 (14.7500%)
running experiment with fault rate and model:  1.0 resnet18
Run 1/1

Test set: Average loss: 5.3635, Accuracy: 1390/10000 (13.9000%)
running experiment with fault rate and model:  5.0 resnet18
Run 1/1

Test set: Average loss: 4.8537, Accuracy: 830/10000 (8.3000%)
running experiment with fault rate and model:  10.0 resnet18

In [14]:
torch.save(masks_list, "masks_list_densenet.pth")

In [20]:
import torch

# Assume G_pos is a 4-d tensor with a specific shape, for example, [2, 3, 64, 8] for demonstration
G_pos = torch.randn(1, 1, 64, 8)

# Initialize the mask with True values
msb_mask = torch.ones_like(G_pos, dtype=torch.bool)

# Calculating the dimensions
dims = list(G_pos.shape)

# Generate a range list for the third dimension, assuming 8 cells per weight bit
range_list = list(range(0, dims[2]//8))

# Iterate over the first two dimensions
for a in range(dims[0]):
    for b in range(dims[1]):
        # Iterate over the calculated range list and set the last 3 bits of each segment to False
        for i in range(len(range_list)):
            msb_mask[a, b, i*8+5:8*i+8] = False

# Verifying the operation
# To verify, let's print out the mask for a specific segment
# This will print the mask values for the segment of the first array and first channel
print(msb_mask[0, 0, :16])  # Check the first 8 cells (representing the first weight bit) to verify the mask

# The expectation is that the last 3 bits of this segment are False, while the others are True



tensor([[ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [F

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import time
import os
import sys
import copy
import math
import matplotlib.pyplot as plt
from resnet import *
from mobilenet import *
from densenet import *
from torx.module.layer import *

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.25.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [15]:
ftrace_list = torch.load("cifar-10/filterwise_trace_densenet.pt")

In [16]:
num_keys = len(ftrace_list)
final_filterwise_trace = [ [sum(values) / num_keys for values in zip(*sublists)]
    for sublists in zip(*ftrace_list.values())
]

In [17]:
print(len(final_filterwise_trace))
print(final_filterwise_trace[0])

121
[0.005786231102374586, 0.008318110023392365, 0.005854737802874297, 0.0231731478284928, 0.02051602901221486, 0.015906232952838762, -2.4449880665997625e-05, 6.135145929874852e-05, 0.0005553520940566159, 0.007846021109435242, 0.009907117469183505, 0.010641722144428059, 0.42524455070495604, 0.432247806340456, 0.3849529208615422, 0.006334291040839162, 0.005275978801728342, 0.0063774528546491635, -0.00021115326249855572, 0.0015069848507118877, -0.0005816457560285926, 0.03346787826914806, 0.03128605586475715, 0.026877140281021637, 0.0010928967694053427, 0.0011502879139152356, 0.00048636208144216654, 0.1740639192610979, 0.1721416874229908, 0.15416671760380268, 0.006064591011963784, 0.005430932816816494, 0.006845232251289417, 0.0007923943563218926, 0.001991431184578687, 0.0015645492792828008, 0.004342789041620563, 0.0042593849072000015, 0.0024290683435538086, 0.0015113729405857156, 0.0007908575566398213, 0.001439287529065041, 0.004427938853768865, 0.004241129518632079, 0.002299760642563342,

In [18]:
torch.save(final_filterwise_trace, "cifar-10/final_filterwise_trace_densenet.pt")